# Secret Spot: Surf Forecasting via Graph-Based Wave Refraction

Prototype implementation for Part 3. Each function has its own cell, followed by a test cell.
Pseudocode from `CONCEPT.md` is included as comments to guide the implementation.


## Imports and constants


In [2]:
import heapq
import math

import numpy as np
import matplotlib.pyplot as plt

G = 9.81  # gravitational acceleration, m/s^2


## Shared conventions

```
GRID:        depth_grid[row][col], depths in meters
             water = positive depth; land = 0 or negative
             row 0 = north edge, col 0 = west edge
CELL:        a (row, col) pair
CELL SIZE:   cell_size = (dy, dx) in meters (north-south, east-west)
DIRECTIONS:  degrees clockwise from north; swell direction = where the swell comes FROM
             (same convention buoys and Surfline use)
```


# 1. Grid building


In [ ]:
def make_synthetic_grid(shape, rows, cols, base_depth, cell_size):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: shape ("flat", "mound", or "slope"), rows and cols (int),
    #        base_depth in meters (float), cell_size (dy, dx) in meters
    # Returns: depth_grid (2D array, rows x cols)
    #
    # if rows < 1 or cols < 1:
    #     raise error "grid must have at least one row and column"
    #
    # create depth_grid of size rows x cols, every cell = base_depth
    #
    # if shape == "flat":
    #     return depth_grid                      # uniform depth everywhere
    #
    # else if shape == "mound":
    #     center = (rows // 2, cols // 2)
    #     radius = min(rows, cols) // 4
    #     peak_depth = base_depth / 4            # mound rises to a quarter of base depth
    #     for each cell (r, c):
    #         d = distance from (r, c) to center, in cells
    #         if d < radius:
    #             # depth shrinks smoothly from base_depth at the edge to peak_depth at the center
    #             depth_grid[r][c] = peak_depth + (base_depth - peak_depth) * (d / radius)
    #     return depth_grid
    #
    # else if shape == "slope":
    #     # shore along the east edge: depth falls linearly from base_depth (west) to 0 (east)
    #     for each cell (r, c):
    #         depth_grid[r][c] = base_depth * (1 - c / (cols - 1))
    #     return depth_grid                      # last column = 0 = land (the beach)
    #
    # else:
    #     raise error "unknown shape"
    raise NotImplementedError


In [ ]:
# Test: make_synthetic_grid
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [ ]:
def load_bathymetry(file_path, bounds):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: file_path (string), bounds = (south, north, west, east) in degrees
    # Returns: depth_grid (2D array), cell_size (dy, dx) in meters
    #
    # read elevation values, latitudes, and longitudes from file
    # keep only points inside bounds
    #
    # if no points remain:
    #     raise error "bounds contain no data"
    #
    # GEBCO and NOAA store ELEVATION: negative below sea level
    # depth_grid = -elevation                    # flip sign so water depth is positive
    #
    # for each cell with a missing value:
    #     set depth to 0                         # treat as land, so no wave path crosses it
    #     record a warning with the count of missing cells
    #
    # convert grid spacing from degrees to meters
    # dy = latitude spacing  * 111,000
    # dx = longitude spacing * 111,000 * cos(average latitude)
    #
    # return depth_grid, (dy, dx)
    raise NotImplementedError


In [ ]:
# Test: load_bathymetry
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [1]:
def wave_speed(depth, period):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: depth in meters (float), period in seconds (float)
    # Returns: wave speed in m/s (float), or NONE for land
    #
    # if period <= 0:
    #     raise error "period must be positive"
    # if depth <= 0:
    #     return NONE                            # land: waves can't travel here
    #
    # deep_wavelength = g * period^2 / (2 * pi)  # wavelength in deep water
    # wavelength = deep_wavelength               # starting guess
    #
    # repeat up to 100 times:
    #     new_wavelength = deep_wavelength * tanh(2 * pi * depth / wavelength)
    #     if |new_wavelength - wavelength| < 0.001:
    #         stop                               # converged
    #     wavelength = new_wavelength
    #
    # return wavelength / period                 # speed = wavelength / period
    raise NotImplementedError


In [ ]:
# Test: wave_speed
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [ ]:
def build_graph(depth_grid, cell_size, period, neighbors):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: depth_grid (2D array), cell_size (dy, dx), period in seconds,
    #        neighbors (8 or 16)
    # Returns: graph (dictionary: cell -> list of (neighbor_cell, travel_time))
    #
    # if neighbors == 8:
    #     offsets = the 8 surrounding cells (up, down, left, right, 4 diagonals)
    # else if neighbors == 16:
    #     offsets = those 8 plus 8 "knight moves" (e.g., 1 down, 2 right)
    # else:
    #     raise error "neighbors must be 8 or 16"
    #
    # compute each cell's speed once, not once per edge
    # speed_grid = wave_speed(depth, period) for every cell
    #
    # graph = empty dictionary
    # for each cell (r, c):
    #     if speed_grid[r][c] is NONE:
    #         continue                           # land cells get no node
    #     graph[(r, c)] = empty list
    #
    #     for each offset (dr, dc):
    #         (nr, nc) = (r + dr, c + dc)
    #         if (nr, nc) is outside the grid or speed_grid[nr][nc] is NONE:
    #             continue
    #         if the move passes through or corners past a land cell:
    #             continue                       # EDGE CASE: no shortcuts through land
    #
    #         distance = sqrt((dr * dy)^2 + (dc * dx)^2)
    #         # average "slowness" of the two cells, so slow shallow cells count fully
    #         travel_time = (distance / 2) * (1 / speed_grid[r][c] + 1 / speed_grid[nr][nc])
    #         append ((nr, nc), travel_time) to graph[(r, c)]
    #
    # return graph
    raise NotImplementedError


In [ ]:
# Test: build_graph
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


# 2. Refraction (Dijkstra)


In [ ]:
def get_source_cells(depth_grid, cell_size, period, swell_direction):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: depth_grid (2D array), cell_size (dy, dx), period in seconds,
    #        swell_direction in degrees (where swell comes FROM)
    # Returns: sources (dictionary: cell -> start_time in seconds);
    #          empty if no water edge faces the swell
    #
    # travel_bearing = (swell_direction + 180) mod 360   # swell travels AWAY from where it comes from
    # travel_north = cos(travel_bearing)
    # travel_east  = sin(travel_bearing)
    #
    # find edge cells where the swell enters the grid
    # candidates = empty set
    # if travel_north < 0: add every cell in the top row (north edge)       # swell heading south
    # if travel_north > 0: add every cell in the bottom row (south edge)    # swell heading north
    # if travel_east  > 0: add every cell in the left column (west edge)    # swell heading east
    # if travel_east  < 0: add every cell in the right column (east edge)   # swell heading west
    # remove land cells from candidates
    #
    # if candidates is empty:
    #     return empty dictionary                # EDGE CASE: swell comes from the land side
    #
    # stagger start times along the wave front
    # reference_speed = average wave_speed(depth, period) over candidates
    # for each cell (r, c) in candidates:
    #     north_position = -r * dy               # rows increase southward
    #     east_position  =  c * dx
    #     position_along_travel[cell] = north_position * travel_north + east_position * travel_east
    #
    # earliest = minimum of position_along_travel
    # sources = empty dictionary
    # for each cell in candidates:
    #     sources[cell] = (position_along_travel[cell] - earliest) / reference_speed
    #
    # return sources
    raise NotImplementedError


In [ ]:
# Test: get_source_cells
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [ ]:
def dijkstra(graph, sources):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: graph (cell -> list of (neighbor, travel_time)),
    #        sources (cell -> start_time)
    # Returns: travel_times (cell -> fastest arrival time, INFINITY if unreached),
    #          predecessors (cell -> previous cell on the fastest path, NONE for sources)
    #
    # --- A. Initialization ---
    # for each cell in graph:
    #     travel_times[cell] = INFINITY
    #     predecessors[cell] = NONE
    # finalized = empty set
    # queue = empty priority queue               # always returns the smallest time first
    #
    # --- B. Load the starting cells ---
    # for each (cell, start_time) in sources:
    #     if cell not in graph:
    #         continue                           # skip sources that are land
    #     travel_times[cell] = start_time
    #     add (start_time, cell) to queue
    #
    # --- C. Main loop ---
    # while queue is not empty:
    #     (time, current) = remove smallest from queue      # C1. extract-min
    #     if current in finalized:
    #         continue                                      # C2. skip outdated queue entries
    #     add current to finalized                          # C3. finalize
    #
    #     for each (neighbor, edge_time) in graph[current]: # C4. relaxation
    #         if neighbor in finalized:
    #             continue
    #         new_time = travel_times[current] + edge_time
    #         if new_time < travel_times[neighbor]:
    #             travel_times[neighbor] = new_time
    #             predecessors[neighbor] = current
    #             add (new_time, neighbor) to queue
    #
    # --- D. Termination ---
    # queue is empty: every reachable cell is finalized;
    # cells still at INFINITY were never reached (blocked by land)
    #
    # return travel_times, predecessors
    raise NotImplementedError


In [ ]:
# Test: dijkstra
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [ ]:
def reconstruct_path(predecessors, travel_times, cell):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: predecessors and travel_times (from dijkstra), cell (row, col)
    # Returns: path (list of cells from offshore source to cell);
    #          empty list if the cell was never reached
    #
    # --- E. Path reconstruction ---
    # if cell not in travel_times or travel_times[cell] == INFINITY:
    #     return empty list                      # EDGE CASE: unreachable cell
    #
    # path = [cell]
    # while predecessors[path's last cell] is not NONE:
    #     append predecessors[path's last cell] to path
    #     if path is longer than the number of cells in the graph:
    #         raise error "cycle in predecessors"   # safety check: should never happen
    #
    # reverse path                               # now reads source -> cell
    # return path
    raise NotImplementedError


In [ ]:
# Test: reconstruct_path
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [ ]:
def count_converging_paths(predecessors, travel_times, pin, radius):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: predecessors and travel_times (from dijkstra),
    #        pin (row, col), radius in cells (int)
    # Returns: count (int) of distinct offshore source cells whose paths end near the pin
    #
    # nearby_water = every cell in travel_times within radius of pin
    #
    # if nearby_water is empty:
    #     raise error "no water near pin: move the pin or increase radius"   # EDGE CASE: pin on land
    #
    # feeding_sources = empty set
    # for each cell in nearby_water:
    #     path = reconstruct_path(predecessors, travel_times, cell)
    #     if path is not empty:
    #         add path[0] to feeding_sources     # path[0] = the offshore starting cell
    #
    # return size of feeding_sources             # 0 means fully blocked from this direction
    raise NotImplementedError


In [ ]:
# Test: count_converging_paths
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [ ]:
def swell_window_profile(depth_grid, cell_size, pin, period, directions, radius, neighbors):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: depth_grid, cell_size, pin, period, directions (list of degrees,
    #        i.e., 0, 10, ..., 350), radius (int), neighbors (8 or 16)
    # Returns: profile (dictionary: direction -> converging path count)
    #
    # graph = build_graph(depth_grid, cell_size, period, neighbors)   # built ONCE
    #
    # profile = empty dictionary
    # for each direction in directions:
    #     sources = get_source_cells(depth_grid, cell_size, period, direction)
    #     if sources is empty:
    #         profile[direction] = 0             # swell from the land side: blocked
    #         continue
    #     travel_times, predecessors = dijkstra(graph, sources)
    #     profile[direction] = count_converging_paths(predecessors, travel_times, pin, radius)
    #
    # return profile
    raise NotImplementedError


In [ ]:
# Test: swell_window_profile
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


# 3. Scoring


In [ ]:
def angle_difference(a, b):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: two directions in degrees (float)
    # Returns: smallest angle between them, 0 to 180 (float)
    #
    # diff = |a - b| mod 360
    # if diff > 180:
    #     diff = 360 - diff                      # EDGE CASE: wrap around north
    # return diff
    raise NotImplementedError


In [ ]:
# Test: angle_difference
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


In [ ]:
def score_conditions(profile, buoy_reading, wind, offshore_direction, cutoffs):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: profile (direction -> path count, from swell_window_profile),
    #        buoy_reading (peak_energy in m²/Hz, peak_period in s, peak_direction in degrees, height in ft),
    #        wind (speed in knots, direction in degrees, where it comes FROM),
    #        offshore_direction in degrees (wind direction that blows offshore at this spot),
    #        cutoffs (energy_min, period_min, exposure_min, light_wind_max, offshore_tolerance)
    # Returns: verdict ("worth checking", "not worth it", or "not enough data"),
    #          reasons (list of strings), swell numbers for display (or NONE)
    #
    # if any buoy_reading or wind value is missing:
    #     return "not enough data", ["missing buoy or wind data"], NONE   # EDGE CASE
    #
    # reasons = empty list
    #
    # --- Is there something? ---
    # nearest_direction = buoy peak_direction rounded to the nearest 10 degrees (360 -> 0)
    # exposed = profile[nearest_direction] >= cutoffs.exposure_min
    # enough_energy = buoy peak_energy >= cutoffs.energy_min
    # long_enough = buoy peak_period >= cutoffs.period_min
    # something = exposed and enough_energy and long_enough
    # for each check that failed: add an explanation to reasons
    #     # i.e., "spot is blocked from this swell direction", "low wave energy"
    #
    # --- Is it clean? ---
    # light_wind = wind speed <= cutoffs.light_wind_max
    # offshore = angle_difference(wind direction, offshore_direction) <= cutoffs.offshore_tolerance
    # clean = light_wind or offshore
    # if not clean: add "onshore or cross-shore wind" to reasons
    #
    # if something and clean:
    #     verdict = "worth checking"
    # else:
    #     verdict = "not worth it"
    #
    # return verdict, reasons, (height, peak_period, peak_direction)
    raise NotImplementedError


In [ ]:
# Test: score_conditions
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)


# 4. Full run


In [ ]:
def run_forecast(pin, bathymetry_file, bounds, buoy_reading, wind, offshore_direction, cutoffs, radius, neighbors):
    """TODO: docstring (inputs, outputs, assumptions)."""
    # Input: pin (row, col), bathymetry_file (string), bounds (south, north, west, east),
    #        buoy_reading, wind, offshore_direction, cutoffs (as in score_conditions),
    #        radius (int), neighbors (8 or 16)
    # Returns: report (verdict, reasons, swell numbers, swell window profile)
    #
    # depth_grid, cell_size = load_bathymetry(bathymetry_file, bounds)
    #
    # if pin is outside the grid:
    #     raise error "pin is outside the bathymetry bounds"           # EDGE CASE
    #
    # period = buoy_reading.peak_period
    # directions = [0, 10, 20, ..., 350]
    #
    # profile = swell_window_profile(depth_grid, cell_size, pin, period, directions, radius, neighbors)
    # verdict, reasons, swell_numbers = score_conditions(profile, buoy_reading, wind, offshore_direction, cutoffs)
    #
    # return report containing verdict, reasons, swell_numbers, profile
    raise NotImplementedError


In [ ]:
# Test: run_forecast
# TODO: add test cases (see Validation and Testing Plan in CONCEPT.md)
